# Calibrazione causale dei pesi

Misura quanto vale ogni asset in **probabilità di vittoria**, con partite a
seed appaiate: la stessa partita giocata due volte, una di controllo e una con
una sola cosa cambiata al setup. Il controllo è condiviso da tutte le
condizioni, quindi una seed costa una partita per condizione.

Perturbazioni sulla sola casa trattata (ruota con la seed):

- **fort**: un fante su un castello neutrale (controllo positivo, atteso ~1/7);
- **castle**: la fortezza di casa diventa un castello (mustering 2→1), la
  direzione legale del gradino castello/fortezza;
- **castle_supply**: `castle` più un'icona supply (l'interazione);
- **power_icon / supply_icon**: +1 icona stampata sulla home;
- **power_token / supply_step / footman / knight**: +1 gettone, +1 supply,
  +1 fante, +1 cavaliere;
- **placebo**: copia identica dei dati (deve dare esattamente 0).

Output in `MyDrive/agot/calibration-ship/`: `seeds.jsonl` (ripresa dopo un
disconnect) e `weights.json` (delta, IC, q, p, unità di vittoria, per casa).

Tempi indicativi su 2 vCPU: ~0,25 s per partita effettiva, quindi 800 seed
(11 partite ciascuna: controllo + 10 condizioni) ≈ 40 minuti.

La cartella calibration-ship è separata dalla vecchia calibration: il setup
ora include la nave lannister nel porto, e riprendere la vecchia run mescolerebbe
due regole diverse. Il runner scrive design.json e rifiuta di riprendere se il
design non coincide.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
if not REPO_DIR.exists():
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/canditoo2/game-of-thrones.git"
    if token:
        url = f"https://{token}@github.com/canditoo2/game-of-thrones.git"
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"],
    check=True,
)
subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
print("repo at", subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip())


In [ ]:
import os
import shutil

%cd /content/game-of-thrones
if shutil.which("cargo") is None:
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!pip install -q -e .


In [ ]:
SEEDS = 800        # minimum pairs per condition
MAX_SEEDS = 2000   # cap when the intervals are still too wide
WORKERS = 2
TARGET = 0.02      # stop extending when the 95% half-width is below this

OUT = "/content/drive/MyDrive/agot/calibration-ship"
!python tools/calibrate_weights.py --data-dir src/agot/data --out {OUT} \
    --seeds {SEEDS} --max-seeds {MAX_SEEDS} --workers {WORKERS} \
    --target-half-width {TARGET}


In [ ]:
import json
from pathlib import Path

result = json.loads(Path("/content/drive/MyDrive/agot/calibration-ship/weights.json").read_text())
print("seeds:", result["design"]["seeds"])
print(
    f"{'condizione':14s} {'n':>5s} {'controllo':>10s} {'trattato':>9s} "
    f"{'delta':>9s} {'q':>6s} {'IC 95%':>18s} {'unita vittoria':>15s}"
)
for name, stats in result["results"].items():
    low, high = stats["ci95"]
    print(
        f"{name:14s} {stats['n']:5d} {stats['p_control'] * 100:9.1f}% "
        f"{stats['p_treated'] * 100:8.1f}% {stats['delta'] * 100:+8.1f}pp "
        f"{stats['q'] * 100:5.1f}% [{low * 100:+5.1f},{high * 100:+5.1f}] "
        f"{stats['victory_units_rules']:+7.2f}"
    )
print("\nper casa:")
for house, conditions in result["by_house"].items():
    supply = conditions.get("supply_step", {})
    fort = conditions.get("fort", {})
    print(
        f"  {house:10s} supply {supply.get('delta', 0) * 100:+5.1f}pp  "
        f"fort {fort.get('delta', 0) * 100:+5.1f}pp"
    )
